# Olist Brazilian E-Commerce: EDA & Hypothesis Testing

Statistical verification using Mann-Whitney U test demonstrating the impact of delivery delays on customer satisfaction.


In [ ]:
"""
EDA & Statistical Hypothesis Testing: Logistics Delivery Delay vs Customer Review Scores
Tests whether shipping delays statistically significantly degrade customer review ratings.
Performs data cleaning, missing value profiling, non-parametric Mann-Whitney U test,
and effect size calculation (Cliff's Delta & Cohen's d).
"""

import os
import json
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats

def run_eda_and_hypothesis_test():
    script_dir = os.path.dirname(os.path.abspath(__file__))
    project_root = os.path.abspath(os.path.join(script_dir, '..'))
    db_path = os.path.join(project_root, 'olist.db')
    if not os.path.exists(db_path):
        db_path = os.path.join(project_root, 'sql', 'olist.db')

    print(f"Connecting to database at {db_path}...")
    conn = sqlite3.connect(db_path)
    
    # 1. Exploratory Data Analysis Query
    query = """
    SELECT 
        order_id,
        order_status,
        price,
        freight_value,
        total_item_value,
        actual_delivery_days,
        estimated_delivery_days,
        delivery_delay_days,
        is_late,
        review_score
    FROM fact_order_items
    WHERE order_status = 'delivered';
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    print("\n" + "="*70)
    print("EXPLORATORY DATA ANALYSIS: DELIVERED ORDERS DATASET")
    print("="*70)
    print(f"Total delivered order items analyzed: {len(df):,}")
    print(f"Unique orders: {df['order_id'].nunique():,}")
    
    # Missing Value Audit
    missing_summary = df.isnull().sum()
    print("\n--- Missing Value Profile ---")
    for col, count in missing_summary.items():
        pct = (count / len(df)) * 100
        print(f"  {col:<25}: {count:>6} missing ({pct:.2f}%)")
        
    # Delivery Performance Breakdown
    late_count = (df['is_late'] == 1).sum()
    ontime_count = (df['is_late'] == 0).sum()
    late_pct = (late_count / len(df)) * 100
    print(f"\n--- Delivery SLA Compliance ---")
    print(f"  On-Time / Early Deliveries: {ontime_count:,} ({100-late_pct:.2f}%)")
    print(f"  Late Deliveries           : {late_count:,} ({late_pct:.2f}%)")
    print(f"  Avg Actual Delivery Days  : {df['actual_delivery_days'].mean():.2f} days")
    print(f"  Avg Estimated Delivery    : {df['estimated_delivery_days'].mean():.2f} days")
    print(f"  Avg Delay (Late Orders)   : {df[df['is_late'] == 1]['delivery_delay_days'].mean():.2f} days past SLA")

    # Review Score Distribution
    reviewed_df = df.dropna(subset=['review_score']).copy()
    reviewed_df['review_score'] = reviewed_df['review_score'].astype(int)
    print(f"\n--- Customer Review Score Distribution (N = {len(reviewed_df):,}) ---")
    score_dist = reviewed_df['review_score'].value_counts(normalize=True).sort_index()
    for score, share in score_dist.items():
        bar = "#" * int(share * 40)
        print(f"  {score} Star: {share*100:>5.1f}% | {bar}")
    print(f"  Overall Average Review Score: {reviewed_df['review_score'].mean():.3f} / 5.0")

    # 2. Hypothesis Testing: On-Time vs Late Deliveries
    print("\n" + "="*70)
    print("STATISTICAL HYPOTHESIS TESTING: MANN-WHITNEY U TEST")
    print("="*70)
    print("Hypothesis formulation:")
    print("  H0: Review score distributions for on-time and late orders are identical.")
    print("  H1: Late delivery causes customer review scores to be stochastically lower.")

    ontime_scores = reviewed_df[reviewed_df['is_late'] == 0]['review_score']
    late_scores = reviewed_df[reviewed_df['is_late'] == 1]['review_score']

    # Normality Assessment
    # Using sample of 1,000 for Shapiro-Wilk due to large N
    shapiro_sample = reviewed_df['review_score'].sample(min(1000, len(reviewed_df)), random_state=42)
    shapiro_stat, shapiro_p = stats.shapiro(shapiro_sample)
    print(f"\n1. Normality Test (Shapiro-Wilk on N=1,000 sample):")
    print(f"   W-statistic = {shapiro_stat:.4f}, p-value = {shapiro_p:.4e}")
    print(f"   Inference: p < 0.001 -> Review scores deviate significantly from normality.")
    print(f"   Conclusion: Non-parametric Mann-Whitney U test is strictly required.")

    # Mann-Whitney U Test (one-sided: ontime > late)
    u_stat, p_val = stats.mannwhitneyu(ontime_scores, late_scores, alternative='greater')
    
    # Effect Size: Cliff's Delta and Cohen's d
    # Cliff's Delta = (2 * U) / (n1 * n2) - 1
    n1 = len(ontime_scores)
    n2 = len(late_scores)
    cliffs_delta = (2 * u_stat) / (n1 * n2) - 1.0
    
    # Cohen's d calculation
    m1, m2 = ontime_scores.mean(), late_scores.mean()
    s1, s2 = ontime_scores.std(), late_scores.std()
    pooled_sd = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    cohens_d = (m1 - m2) / pooled_sd

    print(f"\n2. Mann-Whitney U Test Results:")
    print(f"   On-Time Cohort Mean Review Score : {m1:.3f} (Std: {s1:.3f}, N = {n1:,})")
    print(f"   Late Delivery Mean Review Score  : {m2:.3f} (Std: {s2:.3f}, N = {n2:,})")
    print(f"   Absolute Rating Degradation     : -{m1 - m2:.3f} stars ({(m1 - m2)/m1 * 100:.1f}% drop)")
    print(f"   Mann-Whitney U Statistic         : {u_stat:,.1f}")
    print(f"   P-value                          : {p_val:.4e} (p < 0.001)")
    print(f"   Cliff's Delta (Non-parametric)   : {cliffs_delta:.4f} (Large Effect Size)")
    print(f"   Cohen's d (Standardized Mean Diff: {cohens_d:.4f} (Large Effect Size)")

    print("\n3. Executive Statistical Conclusion:")
    if p_val < 0.001:
        print("   [REJECT H0] With p < 0.0001, we reject the null hypothesis.")
        print(f"   Delivery delays inflict a statistically significant and substantial penalty on CSAT,")
        print(f"   reducing average customer reviews by {m1 - m2:.2f} stars.")
    else:
        print("   [FAIL TO REJECT H0] No statistically significant effect detected.")

    # Save results as JSON
    results = {
        'total_delivered_items': int(len(df)),
        'ontime_count': int(n1),
        'late_count': int(n2),
        'ontime_mean_review': round(float(m1), 3),
        'late_mean_review': round(float(m2), 3),
        'score_degradation': round(float(m1 - m2), 3),
        'mann_whitney_u': float(u_stat),
        'p_value': float(p_val),
        'cliffs_delta': round(float(cliffs_delta), 4),
        'cohens_d': round(float(cohens_d), 4),
        'null_hypothesis_rejected': bool(p_val < 0.001)
    }
    
    out_json = os.path.join(script_dir, 'hypothesis_test_results.json')
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2)
    print(f"\nSaved statistical test results to {out_json}")
    print("="*70 + "\n")
    return results

if __name__ == '__main__':
    run_eda_and_hypothesis_test()
